# E0 (revised) on Kaggle: shuffled CV vs walk-forward, 5 seeds, full-data TabPFN

**Setup (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**, Add-ons → Secrets → `TABPFN_TOKEN` ticked.
No input file is needed: the data downloads automatically.

**Run:** run cell 1 to check (`key loaded: True`, GPU `Tesla T4`), then **Save Version → Save & Run All (Commit)**.
Expected time about 4 to 5 hours (within Kaggle's 12-hour limit), mostly TabPFN on the full training data. When done, download from the version's **Output**:
`e0_table_v2.csv`, `e0_by_window_v2.csv`, `versions.json`, and the full printed log (the notebook itself).

**Consistency check:** the last line prints Random Forest's mean walk-forward AUC over 5 seeds. In E1 (Colab),
Random Forest's honest t=20–29 AUC was 0.8628 and E3's base mean AUC was 0.9458. If the same values appear here,
Kaggle and Colab produce identical results for the classical models.

## 1. Install, key, environment

In [1]:
# %% 1. Install, TabPFN key and environment (Kaggle)
!pip install -q tabpfn==9.2.0
import os, sys, json, glob, importlib.metadata as md
key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("TABPFN_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        key = userdata.get("TABPFN_TOKEN")
    except Exception:
        key = os.environ.get("TABPFN_TOKEN")
if key:
    os.environ["TABPFN_TOKEN"] = key.strip()
    os.environ["TABPFN_NO_BROWSER"] = "1"
print("key loaded:", bool(key), "| length:", len(key.strip()) if key else 0)

versions = {"python": sys.version.split()[0]}
for p in ["numpy", "pandas", "scikit-learn", "xgboost", "tabpfn", "torch", "pyarrow"]:
    try: versions[p] = md.version(p)
    except md.PackageNotFoundError: versions[p] = "not installed"
try:
    import torch
    versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
except ImportError:
    versions["gpu"] = "torch not installed"
print(json.dumps(versions, indent=1))
json.dump(versions, open("versions.json", "w"), indent=1)

key loaded: True | length: 53
{
 "python": "3.13.16",
 "numpy": "2.1.3",
 "pandas": "2.3.3",
 "scikit-learn": "1.6.1",
 "xgboost": "3.4.1",
 "tabpfn": "9.2.0",
 "torch": "2.11.0+cu130",
 "pyarrow": "23.0.1",
 "gpu": "Tesla T4"
}


## 3. Data

In [2]:
# %% 3. Data
import numpy as np, pandas as pd, time, urllib.request
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

if not Path("elliptic_labelled.parquet").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/"
                               "main/resources/elliptic/elliptic_labelled_v2.parquet",
                               "elliptic_labelled.parquet")
ell = pd.read_parquet("elliptic_labelled.parquet")
feat_cols = [c for c in ell.columns if c.startswith("feat_")]
X = ell[feat_cols].values
y = (ell["class"] == 1).astype(int).values
ts = ell["time_step"].values
boundaries = [1, 10, 20, 30, 40, 50]
windows = [(boundaries[i + 1], boundaries[i + 2]) for i in range(len(boundaries) - 2)]
assert len(y) == 46564 and y.sum() == 4545, "Data does not match the dissertation sample"
print("Data check passed:", len(y), "transactions,", int(y.sum()), "illicit")

Data check passed: 46564 transactions, 4545 illicit


## 4. Settings and model configurations (report these in the methods section)

In [3]:
# %% 4. Settings and model configurations (report these in the methods section)
SEEDS = [42, 1, 7, 21, 99]          # same seeds as E3, so E3's base model equals E0's Random Forest
CV_SPLIT_SEED = 42                   # fold assignment is fixed; only model randomness varies by seed
N_BOOT = 1000
PRED_CHUNK = 5000

# Model name -> (random?, list of seeds, training-row cap)
#   "TabPFN"           : full training data (main specification)
#   "TabPFN (10k cap)" : stratified 10,000-row cap, a sensitivity check matching the earlier run.
#                        One seed by default to save GPU time; use SEEDS for all five.
RUNS = {
    "Logistic":         (False, [42], None),
    "Random Forest":    (True,  SEEDS, None),
    "XGBoost":          (True,  SEEDS, None),
    "TabPFN":           (True,  SEEDS, None),
    "TabPFN (10k cap)": (True,  [42],  10000),
}

def make_model(name, seed):
    if name == "Logistic":
        # standardised features, balanced class weights, L2 penalty (default C=1)
        return Pipeline([("sc", StandardScaler()),
                         ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                                                    random_state=42))])
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced",
                                      random_state=seed, n_jobs=-1)
    if name == "XGBoost":
        # 200 trees, depth 5, learning rate 0.1, log-loss objective. Class balance via
        # scale_pos_weight = (licit / illicit) computed on EACH training set (see fit_predict),
        # analogous to class_weight="balanced"; never from the full sample (no look-ahead).
        return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             random_state=seed, eval_metric="logloss", n_jobs=-1)
    if name.startswith("TabPFN"):
        from tabpfn import TabPFNClassifier
        # default configuration, no rebalancing ("no tuning" principle)
        return TabPFNClassifier(device="cuda", random_state=seed)
    raise ValueError(name)

def cap_rows(idx, cap, seed):
    if cap is None or len(idx) <= cap:
        return idx
    rng = np.random.default_rng(seed)
    out = []
    for cls in (0, 1):
        sub = idx[y[idx] == cls]
        out.append(rng.choice(sub, size=int(round(cap * len(sub) / len(idx))), replace=False))
    return np.sort(np.concatenate(out))

def fit_predict(name, seed, cap, tr_idx, te_idx):
    tr_idx = cap_rows(tr_idx, cap, seed)
    m = make_model(name, seed)
    ytr = y[tr_idx]
    if name == "XGBoost":
        m.set_params(scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum())
    m.fit(X[tr_idx], ytr)
    return np.concatenate([m.predict_proba(X[te_idx[k:k + PRED_CHUNK]])[:, 1]
                           for k in range(0, len(te_idx), PRED_CHUNK)])

## 5. Predictions: walk-forward and shuffled 5-fold CV, every model and seed

In [4]:
# %% 5. Predictions: walk-forward and shuffled 5-fold CV, every model and seed
# Saved per model and seed in e0_preds/, so an interrupted run resumes where it stopped.
# Approximate Colab time: Logistic 1 min, Random Forest 15 min, XGBoost 5 min,
# TabPFN about 4 hours (5 seeds, full data), TabPFN 10k cap about 20 min.
Path("e0_preds").mkdir(exist_ok=True)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CV_SPLIT_SEED)
folds = list(skf.split(X, y))
fname = lambda n, s: Path("e0_preds") / f"{n.replace(' ', '_').replace('(', '').replace(')', '')}_seed{s}.npz"

for name, (stoch, seeds, cap) in RUNS.items():
    for s in seeds:
        f = fname(name, s)
        if f.exists():
            continue
        t0 = time.time()
        wf, cv = np.full(len(y), np.nan), np.full(len(y), np.nan)
        for lo, hi in windows:
            tr = np.where(ts < lo)[0]; te = np.where((ts >= lo) & (ts < hi))[0]
            wf[te] = fit_predict(name, s, cap, tr, te)
        for tr, te in folds:
            cv[te] = fit_predict(name, s, cap, tr, te)
        np.savez(f, wf=wf, cv=cv)
        print(f"{name:17s} seed {s:>2}: done ({time.time() - t0:.0f}s)", flush=True)

# Record the TabPFN model checkpoint actually used (model version, distinct from package version)
ckpts = sorted({Path(p).name for p in glob.glob(os.path.expanduser("~/.cache/**/*tabpfn*.safetensors"),
                                                 recursive=True)}
               | {Path(p).name for p in glob.glob("/root/.cache/**/*tabpfn*.safetensors", recursive=True)})
versions["tabpfn_checkpoint"] = ckpts
json.dump(versions, open("versions.json", "w"), indent=1)
print("TabPFN package version:", versions.get("tabpfn"), "| model checkpoint(s):", ckpts)

Logistic          seed 42: done (37s)
Random Forest     seed 42: done (77s)
Random Forest     seed  1: done (77s)
Random Forest     seed  7: done (77s)
Random Forest     seed 21: done (77s)
Random Forest     seed 99: done (77s)
XGBoost           seed 42: done (30s)
XGBoost           seed  1: done (29s)
XGBoost           seed  7: done (29s)
XGBoost           seed 21: done (29s)
XGBoost           seed 99: done (29s)


tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

TabPFN            seed 42: done (3334s)
TabPFN            seed  1: done (3318s)
TabPFN            seed  7: done (3321s)
TabPFN            seed 21: done (3321s)
TabPFN            seed 99: done (3320s)
TabPFN (10k cap)  seed 42: done (871s)
TabPFN package version: 9.2.0 | model checkpoint(s): ['tabpfn-v3.5-20260909.safetensors']


## 6. Results: seed-averaged scores, optimism gaps, error ratio, paired bootstrap CIs

In [5]:
# %% 6. Results: seed-averaged scores, optimism gaps, error ratio, paired bootstrap CIs
from scipy.stats import rankdata

win_idx = [np.where((ts >= lo) & (ts < hi))[0] for lo, hi in windows]
P = {n: [np.load(fname(n, s)) for s in seeds] for n, (_, seeds, _) in RUNS.items()}

def fast_auc(yt, p):
    r = rankdata(p); n1 = yt.sum(); n0 = len(yt) - n1
    return (r[yt == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def scores(name, idx_list, metric):
    """Mean over seeds of the window-averaged metric, for CV and WF predictions."""
    f = fast_auc if metric == "auc" else average_precision_score
    cv = np.mean([np.mean([f(y[i], d["cv"][i]) for i in idx_list]) for d in P[name]])
    wf = np.mean([np.mean([f(y[i], d["wf"][i]) for i in idx_list]) for d in P[name]])
    return cv, wf

def stats(name, idx_list):
    cv_a, wf_a = scores(name, idx_list, "auc")
    cv_p, wf_p = scores(name, idx_list, "ap")
    return {"cv_auc": cv_a, "wf_auc": wf_a, "gap_auc": cv_a - wf_a,
            "cv_ap": cv_p, "wf_ap": wf_p, "gap_ap": cv_p - wf_p,
            "ratio": (1 - wf_a) / (1 - cv_a)}      # how many times larger the true error is

point = {n: stats(n, win_idx) for n in RUNS}

# Paired bootstrap: resample transactions within each test window; the same resample is used for
# CV and WF and for every model and seed, so gaps, ratios and model differences are all paired.
rng = np.random.default_rng(2026)
boot = {n: [] for n in RUNS}
t0 = time.time()
for b in range(N_BOOT):
    bi = [rng.choice(i, size=len(i), replace=True) for i in win_idx]
    for n in RUNS:
        boot[n].append(stats(n, bi))
print(f"bootstrap done ({time.time() - t0:.0f}s)")

ci = lambda v: f"[{np.percentile(v, 2.5):.3f}, {np.percentile(v, 97.5):.3f}]"
rows = []
for n in RUNS:
    B_ = pd.DataFrame(boot[n]); pt = point[n]
    # seed spread of the window-averaged walk-forward AUC
    wf_by_seed = [np.mean([fast_auc(y[i], d["wf"][i]) for i in win_idx]) for d in P[n]]
    rows.append({"Model": n, "Seeds": len(P[n]),
                 "Shuffled AUC": pt["cv_auc"], "Walk-forward AUC": pt["wf_auc"],
                 "WF AUC seed SD": np.std(wf_by_seed, ddof=1) if len(wf_by_seed) > 1 else np.nan,
                 "AUC gap": pt["gap_auc"], "AUC gap 95% CI": ci(B_.gap_auc),
                 "Shuffled AP": pt["cv_ap"], "Walk-forward AP": pt["wf_ap"],
                 "AP gap": pt["gap_ap"], "AP gap 95% CI": ci(B_.gap_ap),
                 "Error ratio": pt["ratio"], "Error ratio 95% CI": ci(B_.ratio)})
e0 = pd.DataFrame(rows).set_index("Model")
pd.set_option("display.width", 250)
print(e0.round(3).to_string())
e0.round(4).to_csv("e0_table_v2.csv")

# Is each model's error ratio larger than Logistic's? (paired bootstrap of the difference)
lr = pd.DataFrame(boot["Logistic"]).ratio.values
print("\nError ratio minus Logistic's (paired bootstrap):")
for n in RUNS:
    if n == "Logistic":
        continue
    d = pd.DataFrame(boot[n]).ratio.values - lr
    print(f"  {n:17s} {point[n]['ratio'] - point['Logistic']['ratio']:+.2f}  95% CI {ci(d)}")

bootstrap done (518s)
                  Seeds  Shuffled AUC  Walk-forward AUC  WF AUC seed SD  AUC gap  AUC gap 95% CI  Shuffled AP  Walk-forward AP  AP gap   AP gap 95% CI  Error ratio Error ratio 95% CI
Model                                                                                                                                                                                 
Logistic              1         0.961             0.903             NaN    0.059  [0.055, 0.062]        0.758            0.550   0.208  [0.195, 0.223]        2.511     [2.380, 2.675]
Random Forest         5         0.993             0.946           0.003    0.047  [0.043, 0.051]        0.962            0.864   0.098  [0.090, 0.105]        7.476     [6.462, 9.006]
XGBoost               5         0.996             0.961           0.000    0.035  [0.032, 0.039]        0.978            0.888   0.089  [0.082, 0.096]        9.912    [7.976, 12.662]
TabPFN                5         0.998             0.968        

## 7. Walk-forward AUC by window (mean and SD over seeds)

In [6]:
# %% 7. Walk-forward AUC by window (mean and SD over seeds)
rows = []
for n in RUNS:
    for (lo, hi), i in zip(windows, win_idx):
        a = [fast_auc(y[i], d["wf"][i]) for d in P[n]]
        rows.append({"Model": n, "Window": f"t={lo}-{hi-1}", "AUC mean": np.mean(a),
                     "AUC SD": np.std(a, ddof=1) if len(a) > 1 else np.nan})
pw = pd.DataFrame(rows)
print(pw.pivot(index="Window", columns="Model", values="AUC mean").round(3).to_string())
print("\nSD across seeds:")
print(pw.pivot(index="Window", columns="Model", values="AUC SD").round(3).to_string())
pw.round(4).to_csv("e0_by_window_v2.csv", index=False)

# Consistency check: with the same seeds and versions, E3's base model IS this Random Forest.
print(f"\nRandom Forest mean walk-forward AUC over 5 seeds: {point['Random Forest']['wf_auc']:.4f}"
      "  (should equal the E3 'base' mean AUC when E3 runs on the same pinned versions)")

Model    Logistic  Random Forest  TabPFN  TabPFN (10k cap)  XGBoost
Window                                                             
t=10-19     0.972          0.995   0.997             0.997    0.996
t=20-29     0.859          0.863   0.917             0.905    0.906
t=30-39     0.890          0.991   0.995             0.995    0.992
t=40-49     0.891          0.935   0.961             0.957    0.948

SD across seeds:
Model    Logistic  Random Forest  TabPFN  TabPFN (10k cap)  XGBoost
Window                                                             
t=10-19       NaN          0.001   0.000               NaN      0.0
t=20-29       NaN          0.010   0.005               NaN      0.0
t=30-39       NaN          0.001   0.000               NaN      0.0
t=40-49       NaN          0.002   0.001               NaN      0.0

Random Forest mean walk-forward AUC over 5 seeds: 0.9458  (should equal the E3 'base' mean AUC when E3 runs on the same pinned versions)
